<a href="https://colab.research.google.com/github/galahad20/assignment_Data_Cleaning/blob/main/ASSIGNMENT_TEMP_DE14_DAY10_DIMAS_ABIAN_IHSAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Data Cleaning
by `Dimas Abian Ihsan`

**Dibimbing - Batch 14 | DE**

## Objectives
- Student mampu melakukan cleaning data
- Student mampu mengatasi missing value
- Student mampu mengimport dan mengecek dataframe
- Student mampu melakukan encoding data
- Student mampu mengatasi outier pada sebuah data

## Persiapan

### Import Library

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

%matplotlib inline
print('Libraries Loaded!')

Libraries Loaded!


### Import Dataset employees.csv menggunakan pandas

In [3]:
employee_path = '/content/drive/MyDrive/dibimbing/day10/dataset/employees.csv'
df_employee = pd.read_csv(employee_path)

In [7]:
df_employee.head()

,employee_id,full_name,email,phone,department,position,salary,hire_date,performance_score,status
0,EMP001,Andi Wijaya,andi.wijaya@company.com,81234567890,IT,Software Engineer,15000000,2022-03-15,85,Active
1,EMP002,Budi Santoso,budi.santoso@company,6282345678901,Finance,Accountant,12000000,2021-06-20,78,Active
2,EMP003,Citra Dewi,NaN,83456789012,HR,HR Manager,18000000,2020-01-10,92,Active
3,EMP004,Dian Pratama,dian.pratama@company.com,6284567890123,IT,Data Analyst,14000000,2023-02-28,88,Active
4,EMP005,Eka Putri,eka@company.com,6285678901234,Marketing,Marketing Executive,11000000,2022-08-15,-5,Active


## Pemahaman Dataset

### Tampilkan info dan statistik desriptive

In [8]:
df_employee.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   employee_id        31 non-null     object
 1   full_name          31 non-null     object
 2   email              26 non-null     object
 3   phone              31 non-null     int64 
 4   department         31 non-null     object
 5   position           31 non-null     object
 6   salary             31 non-null     int64 
 7   hire_date          31 non-null     object
 8   performance_score  31 non-null     int64 
 9   status             31 non-null     object
dtypes: int64(3), object(7)
memory usage: 2.6+ KB


Deksripsi Kolom
- employee-id : tanda pengenal yang dimiliki setiap employee. Sifatnya unqiue
- full_name : nama lengkap employee
- email : email milik employee
- phone : nomor telpon aktif karyawan yang dapat dihubungi
- department : Unit kerja atau departemen tempat karyawan bernaung (misal: IT, HR, Finance).
- position : jabatan employee
- salary : gaji employee
- hire_date : Tanggal resmi karyawan mulai bekerja di perusahaan
- performance_score : Nilai evaluasi kinerja karyawan yang biasanya diberikan dalam skala numerik atau kategori.
- status: status keaktifan karyawan

In [9]:
df_employee.describe()

,phone,salary,performance_score
count,3.100000e+01,3.100000e+01,31.000000
mean,3.283022e+12,1.312903e+07,80.580645
std,3.149403e+12,2.918241e+06,16.863717
min,8.111122e+10,8.500000e+06,-5.000000
25%,8.238900e+10,1.100000e+07,78.500000
50%,6.280123e+12,1.300000e+07,83.000000
75%,6.282340e+12,1.500000e+07,87.500000
max,6.289012e+12,2.000000e+07,95.000000


## Handle DATA DUPLICATE
Di assignment tidak disebutkan, namun terdapat data duplikat yang harus dibersihkkan sebelum ke data cleaning lainnya.

In [10]:
df_employee.duplicated().sum()

np.int64(1)

In [11]:
#tampil data duplicate
df_employee[df_employee.duplicated()]

,employee_id,full_name,email,phone,department,position,salary,hire_date,performance_score,status
30,EMP001,Andi Wijaya,andi.wijaya@company.com,81234567890,IT,Software Engineer,15000000,2022-03-15,85,Active


In [12]:
#hapus data duplicate
df_employee.drop_duplicates(inplace=True)

#cek lagi
df_employee.duplicated().sum()

np.int64(0)

## Penanganan Missing Value

### Cek missing value

In [13]:
df_employee.isna().sum()

,0
employee_id,0
full_name,0
email,5
phone,0
department,0
position,0
salary,0
hire_date,0
performance_score,0
status,0


In [14]:
df_employee[df_employee['email'].isnull()]

,employee_id,full_name,email,phone,department,position,salary,hire_date,performance_score,status
2,EMP003,Citra Dewi,NaN,83456789012,HR,HR Manager,18000000,2020-01-10,92,Active
7,EMP008,Hendra Kusuma,NaN,88901234567,HR,Recruiter,10000000,2023-05-15,75,Active
13,EMP014,Nanda Putra,NaN,6281444555666,Marketing,Digital Marketing,11500000,2021-08-20,80,Active
20,EMP021,Umi Kalsum,NaN,6282111222333,HR,Payroll Staff,9000000,2023-04-10,74,Active
27,EMP028,Bayu Segara,NaN,6282888999000,IT,System Administrator,14000000,2021-09-15,84,Active


Menurut saya, data yang hilang ini bertipe MCAR (Missing Completely At Random) karena datanya sendiri tidak menunjukkan pola tertentu yang depends on specific variabel. Belum lagi tidak ditemukan alasan seperti pegawai baru maupun department tertentu sebab ada 3 department berbeda. Juga tidak ditemukan indikasi bahwa data sengaja tidak diisi karena alasan sensitivitas nilai email itu sendiri atau alasan internal lainnya yang bersifat sistematis.


Dari hasil identifikasi, saya memperoleh 5 data yang 'email'nya missing dari 31 data. Ini berarti sekitar 16% dari keseluruhan data.

Maka metode yang saya lakukan adalah imputasi.
Berdasarkan pada struktur email di baris lain, saya menemukan pola penulisan email adalah 'full_name@company.com'

Maka nanti email yang kosong akan diisi dengan format di atas.


### Penanganan null

In [15]:
#copy dulu agar tidak merusak df sebelumnya
df_employee_clean = df_employee.copy()

df_employee_clean.head()

,employee_id,full_name,email,phone,department,position,salary,hire_date,performance_score,status
0,EMP001,Andi Wijaya,andi.wijaya@company.com,81234567890,IT,Software Engineer,15000000,2022-03-15,85,Active
1,EMP002,Budi Santoso,budi.santoso@company,6282345678901,Finance,Accountant,12000000,2021-06-20,78,Active
2,EMP003,Citra Dewi,NaN,83456789012,HR,HR Manager,18000000,2020-01-10,92,Active
3,EMP004,Dian Pratama,dian.pratama@company.com,6284567890123,IT,Data Analyst,14000000,2023-02-28,88,Active
4,EMP005,Eka Putri,eka@company.com,6285678901234,Marketing,Marketing Executive,11000000,2022-08-15,-5,Active


In [16]:
#imputasi email yang null
# buat function untuk imputasi email yg hilang dengan nama dan format yang sesuai dengan email lainnya
def generate_email(full_name):
    df_employee_clean['full_name'] = df_employee_clean['full_name'].str.strip()
    temp = str(full_name).lower().split(' ') #buat list baru untuk menyimpan nama yang telah dipisah
    if len(temp) >= 2:
        return f"{temp[0]}.{temp[1]}@company.com"
    else:
        return f"{temp[0]}@company.com"

df_employee_clean['email'] = df_employee_clean['email'].fillna(df_employee_clean['full_name'].apply(generate_email))

### Cek kembali hasil

In [17]:
#cek kembali jumlah yang null tadi
df_employee_clean.isna().sum()

,0
employee_id,0
full_name,0
email,0
phone,0
department,0
position,0
salary,0
hire_date,0
performance_score,0
status,0
